# GuindiMa — test de prononciation de la voix Oolel native

Voix : celle du **Space soynade-research/Oolel-Voices-Demo**, fichier `8_1_c.wav`
(c'est la voix de référence que la documentation d'Oolel-Voices conseille
elle-même). Aucune voix Adia : tout se télécharge, rien à importer.

Un seul fichier par variante (3 essais, Whisper wolof garde le meilleur,
graine 1234) — les mêmes réglages que les autres séries.

**Avant :** GPU T4 · clé 🔑 `HF_TOKEN` dans les secrets Colab (pour Whisper).

**Lancer :** Exécution → Tout exécuter. La 1re cellule redémarre la session
(« session plantée » = normal). Clique sur la **2e cellule** puis **Ctrl + F10**.

**À l'écoute :** les audios arrivent dans `/content/drive/MyDrive/guindima_test_prononciation/audios/`.
- « biis / bis », « Taaf Taaf / Taf Taf / Tafe Tafe », « Akapes / Akaapes »,
  « Albeer Saróo / Alber Saroo », « Aseknaa / Asekna » : noter la lecture de
  « ee », « aa », la forme de Wakaam…
- Puis remplir le lexique `src/tts/lexique_prononciation.py` (clé = écran,
  valeur = orthographe que la voix sait dire), regénérer la liste complète
  (`scripts/preparer_regen_oolel.py`) et le notebook des 869 audios
  (`scripts/preparer_notebook_oolel.py`).


In [ ]:
!pip -q install 'librosa>=0.10.2' transformers==4.46.3 diffusers==0.29.0 conformer==0.3.2 torchcodec s3tokenizer num2words soundfile
import os; os.kill(os.getpid(), 9)  # redémarrage volontaire

In [ ]:
# La voix de référence se télécharge depuis le Space Oolel-Voices-Demo
import os, glob, shutil
from google.colab import drive
drive.mount('/content/drive')
BASE = '/content/drive/MyDrive/guindima_test_prononciation'
SORTIE = f'{BASE}/audios'
RAPPORT = f'{BASE}/rapport_test_prononciation.csv'
REF = f'{BASE}/8_1_c.wav'
os.makedirs(SORTIE, exist_ok=True)
if not os.path.exists(REF):
    from huggingface_hub import hf_hub_download
    telecharge = hf_hub_download(
        repo_id='soynade-research/Oolel-Voices-Demo',
        repo_type='space',
        filename='8_1_c.wav',
    )
    shutil.copy(telecharge, REF)
print('voix de référence :', REF)
print('déjà faits :', len(glob.glob(f'{SORTIE}/*.wav')), '/ 12')


In [ ]:
PLAN = {'modele': 'soynade-research/Oolel-Voices',
 'voix_reference': '8_1_c.wav (Space soynade-research/Oolel-Voices-Demo)',
 'graine': 1234,
 'phrases': [{'fichier': 'test_biis.wav', 'phrase': 'Baal ma, biis yi rekk laa xam.', 'type': 'test'},
             {'fichier': 'test_bis.wav', 'phrase': 'Baal ma, bis yi rekk laa xam.', 'type': 'test'},
             {'fichier': 'test_biis_taaf_taaf_wakaam.wav',
              'phrase': 'Jëlal biis bu Dem Dikk, Taaf Taaf Wakaam.',
              'type': 'test'},
             {'fichier': 'test_bis_taf_taf_wakaam.wav',
              'phrase': 'Jëlal bis bu Dem Dikk, Taf Taf Wakaam.',
              'type': 'test'},
             {'fichier': 'test_biis_tafe_tafe_wakaam.wav',
              'phrase': 'Jëlal biis bu Dem Dikk, Tafe Tafe Wakaam.',
              'type': 'test'},
             {'fichier': 'test_akapes.wav', 'phrase': 'Akapes', 'type': 'test'},
             {'fichier': 'test_akaapes.wav', 'phrase': 'Akaapes', 'type': 'test'},
             {'fichier': 'test_albeer_saroo.wav', 'phrase': 'Albeer Saróo', 'type': 'test'},
             {'fichier': 'test_alber_saroo.wav', 'phrase': 'Alber Saroo', 'type': 'test'},
             {'fichier': 'test_aseknaa.wav', 'phrase': 'Aseknaa', 'type': 'test'},
             {'fichier': 'test_asekna.wav', 'phrase': 'Asekna', 'type': 'test'},
             {'fichier': 'test_maalekum.wav', 'phrase': 'Maalekum salaam ! Fan nga bëgga dem ?', 'type': 'test'}]}
PHRASES = PLAN['phrases']
GRAINE = PLAN['graine']
print(len(PHRASES), 'variantes à écouter')


In [ ]:
# HELPERS_VERIFICATION_VOIX
# Ces fonctions sont pures (ni torch ni numpy) : les tests locaux exécutent
# directement cette cellule pour vérifier le choix du meilleur candidat.
import re
import unicodedata


SEUIL_SIMILARITE = 0.5
VITESSE_CAR_PAR_SEC = 12.0
TOLERANCE_MIN = 0.8
TOLERANCE_RATIO = 0.6
DUREE_MIN = 0.45


def normaliser_texte(texte):
    """Minuscules, sans accents ni ponctuation, pour comparer Whisper au plan."""
    texte = (texte or "").lower()
    texte = texte.translate(str.maketrans({"ŋ": "n", "ë": "e", "ñ": "n"}))
    texte = unicodedata.normalize("NFKD", texte)
    texte = "".join(c for c in texte if not unicodedata.combining(c))
    return re.sub(r"[^a-z0-9]+", " ", texte).strip()


def distance_levenshtein(a, b):
    """Distance de caractères classique, en O(min(len(a), len(b))) mémoire."""
    if len(a) < len(b):
        a, b = b, a
    precedente = list(range(len(b) + 1))
    for i, caractere_a in enumerate(a, 1):
        courante = [i]
        for j, caractere_b in enumerate(b, 1):
            cout = 0 if caractere_a == caractere_b else 1
            courante.append(min(
                courante[-1] + 1,
                precedente[j] + 1,
                precedente[j - 1] + cout,
            ))
        precedente = courante
    return precedente[-1]


def similarite(phrase, transcription):
    """Score de 0 à 1 : 1 = transcription identique après normalisation."""
    a = normaliser_texte(phrase)
    b = normaliser_texte(transcription)
    if not a and not b:
        return 1.0
    return 1.0 - distance_levenshtein(a, b) / max(len(a), len(b))


def bornes_duree(phrase):
    """Bornes basses et hautes attendues pour une phrase, en secondes."""
    longueur = len(normaliser_texte(phrase))
    attendu = max(DUREE_MIN, longueur / VITESSE_CAR_PAR_SEC)
    tolerance = max(TOLERANCE_MIN, attendu * TOLERANCE_RATIO)
    return max(DUREE_MIN, attendu - tolerance), attendu + tolerance


def controler_duree(duree_s, phrase):
    """Renvoie (ok, raison) ; la borne haute attrape le son après la fin."""
    mini, maxi = bornes_duree(phrase)
    if duree_s < mini:
        return False, "trop court"
    if duree_s > maxi:
        return False, "trop long / son après la fin attendue"
    return True, ""


def est_douteux(score, duree_ok):
    """Douteux si Whisper doute ou si la durée est hors bornes."""
    return score < SEUIL_SIMILARITE or not duree_ok


def choisir_meilleur_candidat(candidats):
    """Indice du meilleur candidat : similarité, puis durée valide.

    Un candidat trop court ou trop long est pénalisé de moitié : mieux vaut
    une phrase un peu moins bien reconnue qu'un WAV coupé ou prolongé.
    """
    if not candidats:
        raise ValueError("aucun candidat")

    def cle(indice):
        candidat = candidats[indice]
        ajuste = candidat["score"] if candidat.get("duree_ok", True) else candidat["score"] * 0.5
        return ajuste, candidat["score"], -indice

    return max(range(len(candidats)), key=cle)


In [ ]:
import torch
device = 'cuda:0'
assert torch.cuda.is_available(), 'Active le GPU T4'
# Whisper de contrôle : notre fine-tuning Dakar d'abord, repli public ensuite.
import os

import numpy as np
from transformers import WhisperForConditionalGeneration, WhisperProcessor

WHISPER_PRINCIPAL = "PythonMindset/whisper-small-wolof-dakar-essai3"
WHISPER_SECOURS = "M9and2M/whisper-small-wolof"

# Secret Colab ; hors Colab, on accepte aussi la variable d'environnement.
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN")
HF_TOKEN = (HF_TOKEN or "").strip() or None

if HF_TOKEN is None:
    print("Secret HF_TOKEN absent : repli sur", WHISPER_SECOURS)


def charger_whisper():
    for depot, jeton in ((WHISPER_PRINCIPAL, HF_TOKEN), (WHISPER_SECOURS, None)):
        try:
            processeur = WhisperProcessor.from_pretrained(depot, token=jeton)
            modele = WhisperForConditionalGeneration.from_pretrained(depot, token=jeton).to(device)
            modele.eval()
            print("Whisper de contrôle :", depot)
            return processeur, modele, depot
        except Exception as exc:
            print(f"[Whisper] {depot} inaccessible ({type(exc).__name__} : {exc})")
    raise RuntimeError(f"ni {WHISPER_PRINCIPAL} ni {WHISPER_SECOURS} ne se chargent")


asr_proc, asr_model, asr_depot = charger_whisper()


def reechantillonner_16k(audio, sr):
    """Whisper attend 16 kHz mono ; interpolation linéaire sans dépendance."""
    audio = np.asarray(audio, dtype=np.float32).squeeze()
    if audio.ndim > 1:
        audio = audio.mean(axis=1)
    if sr == 16000 or audio.size == 0:
        return audio
    n = int(round(len(audio) * 16000 / sr))
    if n <= 1:
        return audio[:n]
    x_ancien = np.linspace(0.0, 1.0, len(audio), endpoint=False, dtype=np.float64)
    x_nouveau = np.linspace(0.0, 1.0, n, endpoint=False, dtype=np.float64)
    return np.interp(x_nouveau, x_ancien, audio).astype(np.float32)


def transcrire(audio, sr):
    """Transcrit un WAV déjà généré (pas de language : « wo » fait planter Whisper)."""
    audio_16k = reechantillonner_16k(audio, sr)
    entree = asr_proc(audio_16k, sampling_rate=16000, return_tensors="pt")
    with torch.inference_mode():
        ids = asr_model.generate(entree.input_features.to(device), max_new_tokens=96)
    return asr_proc.batch_decode(ids, skip_special_tokens=True)[0].strip()


In [ ]:
# Oolel-Voices
import torchaudio
from huggingface_hub import snapshot_download
from transformers import AutoModel
oolel = AutoModel.from_pretrained(snapshot_download('soynade-research/Oolel-Voices'), trust_remote_code=True)
SR = oolel.sr
print('Oolel chargé | fréquence :', SR)

In [ ]:
# Générer : 3 essais par audio, Whisper choisit le meilleur. Reprise automatique.
import csv, time
import numpy as np
import soundfile as sf

CHAMPS = ['fichier', 'phrase', 'score_1', 'score_2', 'score_3', 'duree_s', 'retenu', 'douteux']
GRAINES = [GRAINE, GRAINE + 1, GRAINE + 2]

def nettoyer(audio, sr, marge=0.05, cible=0.97):
    audio = np.asarray(audio, dtype=np.float32).squeeze()
    if audio.ndim > 1:
        audio = audio.mean(axis=0)
    if audio.size == 0:
        return audio
    seuil = max(1e-3, 0.02 * float(np.max(np.abs(audio))))
    forts = np.where(np.abs(audio) > seuil)[0]
    if forts.size:
        audio = audio[max(0, forts[0] - int(sr * marge)):min(len(audio), forts[-1] + int(sr * marge))]
    crete = float(np.max(np.abs(audio))) or 1.0
    return (audio * (cible / crete)).astype(np.float32)

def generer(phrase, graine):
    torch.manual_seed(graine); torch.cuda.manual_seed_all(graine)
    with torch.inference_mode():
        wav = oolel.generate(phrase, audio_prompt_path=REF, cfg_weight=0.5, exaggeration=0.2, temperature=0.3)
    wav = wav.detach().float().cpu().numpy() if hasattr(wav, 'cpu') else np.asarray(wav)
    return nettoyer(wav, SR)

lignes = []
if os.path.exists(RAPPORT):
    with open(RAPPORT, encoding='utf-8-sig') as f:
        lignes = list(csv.DictReader(f))
faits = {l['fichier'] for l in lignes}
a_faire = [p for p in PHRASES if not (p['fichier'] in faits and os.path.exists(f"{SORTIE}/{p['fichier']}"))]
print(len(a_faire), 'audios à générer')
debut = time.perf_counter()
for n, item in enumerate(a_faire, 1):
    cands = []
    for g in GRAINES:
        audio = generer(item['phrase'], g)
        duree = len(audio) / SR
        duree_ok, _ = controler_duree(duree, item['phrase'])
        try:
            score = similarite(item['phrase'], transcrire(audio, SR))
        except Exception:
            score = 0.0
        cands.append({'graine': g, 'audio': audio, 'duree': duree, 'score': score, 'duree_ok': duree_ok})
    i = choisir_meilleur_candidat(cands)
    r = cands[i]
    sf.write(f"{SORTIE}/{item['fichier']}", r['audio'], SR, subtype='PCM_16')
    lignes = [l for l in lignes if l['fichier'] != item['fichier']] + [{
        'fichier': item['fichier'], 'phrase': item['phrase'],
        'score_1': round(cands[0]['score'], 3), 'score_2': round(cands[1]['score'], 3), 'score_3': round(cands[2]['score'], 3),
        'duree_s': round(r['duree'], 2), 'retenu': f"{i + 1} (graine {r['graine']})",
        'douteux': 'oui' if est_douteux(r['score'], r['duree_ok']) else 'non'}]
    with open(RAPPORT, 'w', encoding='utf-8-sig', newline='') as f:
        w = csv.DictWriter(f, fieldnames=CHAMPS); w.writeheader(); w.writerows(lignes)
    reste = (time.perf_counter() - debut) / n * (len(a_faire) - n) / 60
    print(f"[{n}/{len(a_faire)}] {item['fichier']} score={r['score']:.2f} | reste ≈ {reste:.0f} min")
print('Terminé.')

In [ ]:
# Écouter et télécharger le zip
from google.colab import files
manquants = [p['fichier'] for p in PHRASES if not os.path.exists(f"{SORTIE}/{p['fichier']}")]
assert not manquants, f"{len(manquants)} audios manquants : relance la cellule de génération."
shutil.copy(RAPPORT, f'{SORTIE}/rapport_test_prononciation.csv')
shutil.make_archive('/content/test_prononciation', 'zip', SORTIE)
shutil.copy('/content/test_prononciation.zip', f'{BASE}/test_prononciation.zip')
print('12 variantes OK — à écouter une par une, puis remplir le lexique.')
files.download('/content/test_prononciation.zip')
